In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import (
    ChatPromptTemplate,
    PromptTemplate,
    FewShotChatMessagePromptTemplate
)
from langchain_core.output_parsers import StrOutputParser

load_dotenv()


/opt/anaconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

# Type 1 : PromptTemplate
## plain string - no roles no system messages
## Simple and old-school — still useful for quick tasks

In [5]:
llm=ChatGroq(model="llama-3.1-8b-instant")
parser=StrOutputParser()

# type1 prompt template 
simple_prompt=PromptTemplate.from_template("Write a one-line tagline for a startup called {name}"
    "that builds {product}."
)
filled=simple_prompt.invoke({"name":"Neurastack","product":"AI tools for developer"})
print("── What Type 1 sends to LLM ──")
print(filled)   
print()
simple_chain=simple_prompt|llm|parser
result = simple_chain.invoke({
    "name": "NeuraStack",
    "product": "AI tools for developers"
})
print("── Type 1 output ──")
print(result)
print()

── What Type 1 sends to LLM ──
text='Write a one-line tagline for a startup called Neurastackthat builds AI tools for developer.'

── Type 1 output ──
"Empowering developers to build a brighter future, one neural connection at a time."



# Type 2 : ChatPromptTemplate
## Structured conversation with roles
## system = LLM personality/instructions
## human  = user question (with variables)

In [7]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a {role}. "
     "Always respond in under 3 sentences. "
     "Never use bullet points."),
    ("human", "{question}")
])
filled_chat = chat_prompt.invoke({
    "role": "startup founder giving advice",
    "question": "How do I get my first 10 customers?"
})
print("── What Type 2 sends to LLM ──")
for msg in filled_chat.messages:
    print(f"{msg.type.upper()}: {msg.content}")
print()
chat_chain = chat_prompt | llm | parser
print("── Type 2: role = startup founder ──")
print(chat_chain.invoke({
    "role": "startup founder giving advice",
    "question": "How do I get my first 10 customers?"
}))
print()

print("── Type 2: role = professor of business ──")
print(chat_chain.invoke({
    "role": "professor of business",
    "question": "How do I get my first 10 customers?"
}))
print()


── What Type 2 sends to LLM ──
SYSTEM: You are a startup founder giving advice. Always respond in under 3 sentences. Never use bullet points.
HUMAN: How do I get my first 10 customers?

── Type 2: role = startup founder ──
Focus on building a minimum viable product and validate your idea with a small group of friends, family, or potential customers. Reach out to your personal network and attend local events, conferences, or meetups to spread the word about your product and generate interest.

── Type 2: role = professor of business ──
To get your first 10 customers, focus on building a strong online presence through social media and a professional website. Reach out to friends, family, and colleagues to spread the word about your business and encourage them to share their experiences with others.



# Type 3 : FewShotChatMessagePromptTemplate
## Show the LLM examples of good input → output
## Better than just telling it what to do

In [8]:
examples = [
    {
        "input": "The product broke after 2 days. Very disappointed.",
        "output": "Negative"
    },
    {
        "input": "Absolutely love this! Best purchase I've made.",
        "output": "Positive"
    },
    {
        "input": "It's okay. Nothing special but gets the job done.",
        "output": "Neutral"
    },
    {
        "input": "Waste of money. Stopped working immediately.",
        "output": "Negative"
    },
]

# Template for each example — how one example looks
example_template = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai",    "{output}")
])
few_shot = FewShotChatMessagePromptTemplate(
    example_prompt=example_template,
    examples=examples
)
final_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a sentiment classifier. "
     "Classify the sentiment of the review as: "
     "Positive, Negative, or Neutral. "
     "Respond with ONLY one word."),
    few_shot,               # ← all 4 examples injected here
    ("human", "{review}")   # ← actual input to classify
])
# Inspect what the full prompt looks like
filled_few_shot = final_prompt.invoke({
    "review": "This is the worst app I have ever used."
})
print("── What Type 3 sends to LLM ──")
for msg in filled_few_shot.messages:
    print(f"{msg.type.upper()}: {msg.content}")
print()
few_shot_chain = final_prompt | llm | parser
test_reviews = [
    "This changed my life. I use it every single day.",
    "Meh. It works sometimes.",
    "Terrible. Would not recommend to anyone.",
    "Pretty good overall, a few minor issues.",
]
print("── Type 3 outputs ──")
for review in test_reviews:
    sentiment = few_shot_chain.invoke({"review": review})
    print(f"Review : {review}")
    print(f"Verdict: {sentiment}")
    print()

── What Type 3 sends to LLM ──
SYSTEM: You are a sentiment classifier. Classify the sentiment of the review as: Positive, Negative, or Neutral. Respond with ONLY one word.
HUMAN: The product broke after 2 days. Very disappointed.
AI: Negative
HUMAN: Absolutely love this! Best purchase I've made.
AI: Positive
HUMAN: It's okay. Nothing special but gets the job done.
AI: Neutral
HUMAN: Waste of money. Stopped working immediately.
AI: Negative
HUMAN: This is the worst app I have ever used.

── Type 3 outputs ──
Review : This changed my life. I use it every single day.
Verdict: Positive

Review : Meh. It works sometimes.
Verdict: Neutral

Review : Terrible. Would not recommend to anyone.
Verdict: Negative

Review : Pretty good overall, a few minor issues.
Verdict: Neutral

